In [ ]:
import dataclasses

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

from gbp.loaders.contracts import GraphLoaderConfig
from gbp.loaders.dataloader_graph import DataLoaderGraph
from gbp.loaders.dataloader_mock import DataLoaderMock
from gbp.build.pipeline import build_model

mock_config = {
    "n_stations": 10,
    "n_depots": 2,
    "n_timestamps": 72,
    "time_freq": "h",
    "start_date": "2025-01-01",
    "ebike_fraction": 0.3,
    "depot_capacity": 200,
    "seed": 42,
}
mock_source = DataLoaderMock(mock_config, n_trucks=3, truck_capacity_bikes=20)
graph_loader = DataLoaderGraph(mock_source, GraphLoaderConfig())
raw_model = graph_loader.load()
resolved_with_supply = build_model(raw_model)
resolved_model = dataclasses.replace(resolved_with_supply, supply=None)

2026-05-16 11:53:42 [info     ] load_start                     loader=graph_core
2026-05-16 11:53:42 [debug    ] source_validated               loader=graph_core
2026-05-16 11:53:42 [debug    ] observed_flow_built            loader=graph_core rows=296
2026-05-16 11:53:42 [debug    ] observed_inventory_built       loader=graph_core rows=60
2026-05-16 11:53:42 [info     ] load_done                      facilities=12 loader=graph_core


In [7]:
### Initially we should have only df_trips
### Some of the tables we can generate using df_trips
### Some of the tables we can generate using mock_functions
### Thus - we have two types of tables

# self.df_stations
# self.df_station_capacities
# self.df_station_costs, 

# self.df_depots
# self.df_depot_capacities
# self.df_depot_costs, 

# self.timestamps

# self.inventory_initial,
# self.df_telemetry_ts, 
# self.df_trips

# self.df_resources
# self.df_resource_capacities
# self.df_truck_rates

In [8]:
mock_source.timestamps

DatetimeIndex(['2025-01-01 00:00:00', '2025-01-01 01:00:00',
               '2025-01-01 02:00:00', '2025-01-01 03:00:00',
               '2025-01-01 04:00:00', '2025-01-01 05:00:00',
               '2025-01-01 06:00:00', '2025-01-01 07:00:00',
               '2025-01-01 08:00:00', '2025-01-01 09:00:00',
               '2025-01-01 10:00:00', '2025-01-01 11:00:00',
               '2025-01-01 12:00:00', '2025-01-01 13:00:00',
               '2025-01-01 14:00:00', '2025-01-01 15:00:00',
               '2025-01-01 16:00:00', '2025-01-01 17:00:00',
               '2025-01-01 18:00:00', '2025-01-01 19:00:00',
               '2025-01-01 20:00:00', '2025-01-01 21:00:00',
               '2025-01-01 22:00:00', '2025-01-01 23:00:00',
               '2025-01-02 00:00:00', '2025-01-02 01:00:00',
               '2025-01-02 02:00:00', '2025-01-02 03:00:00',
               '2025-01-02 04:00:00', '2025-01-02 05:00:00',
               '2025-01-02 06:00:00', '2025-01-02 07:00:00',
               '2025-01-

In [9]:
### Выпиши все таблицы!!!

# temporal = self._build_temporal()
# entities = self._build_entities()
# behavior = self._build_behavior(entities)
# distance_data = self._build_distance_matrix(entities) if self._config.build_edges else {}
# resources = self._build_resources(entities)

# observations: dict[str, pd.DataFrame | None] = {}
# if self._config.build_observations:
#     observations = self._build_observations(entities)

# registry = AttributeRegistry()
# node_params = self._build_node_parameters(registry)
# self._register_facility_costs(registry, temporal)
# self._register_resource_costs(registry, entities)

# all_tables = {
#     **temporal,
#     **entities.tables,
#     **behavior,
#     **distance_data,
#     **node_params,
#     **resources,
#     **{k: v for k, v in observations.items() if v is not None},
# }
# return RawModelData(
#     **{k: v for k, v in all_tables.items() if v is not None},
#     attributes=registry,
# )

In [10]:
import pandas as pd
import requests


# Trips data
# trips_df = pd.read_csv("E:\\Documents\\vlzm\\GFDRR\\data\\raw\\202602-citibike-tripdata_1.csv")

trips_df =  pd.read_csv("/Users/vladislav/Documents/vlzm/GFDRR/data/raw/202601-citibike-tripdata_1.csv")
# Stations data
GBFS_BASE = "https://gbfs.citibikenyc.com/gbfs/en"
station_info = requests.get(f"{GBFS_BASE}/station_information.json").json()
station_status = requests.get(f"{GBFS_BASE}/station_status.json").json()
info_df = pd.DataFrame(station_info["data"]["stations"])
status_df = pd.DataFrame(station_status["data"]["stations"])
stations = info_df.merge(status_df, on="station_id")

# self.df_stations
df_stations_start = trips_df[['start_station_id', 'start_lat', 'start_lng']].drop_duplicates(subset=['start_station_id'])
df_stations_start = df_stations_start.rename(columns = {'start_station_id': 'station_id', 'start_lat': 'lat', 'start_lng': 'lng'})
df_stations_end = trips_df[['end_station_id', 'end_lat', 'end_lng']].drop_duplicates(subset=['end_station_id'])
df_stations_end = df_stations_end.rename(columns = {'end_station_id': 'station_id', 'end_lat': 'lat', 'end_lng': 'lng'})
df_stations = pd.concat([df_stations_start, df_stations_end]).reset_index(drop = True)
df_stations = df_stations.dropna(subset=['station_id']).drop_duplicates(subset=['station_id']).reset_index(drop=True)

# self.df_station_capacities
df_stations_capacities = stations[stations['is_installed']==1][['short_name', 'capacity']].drop_duplicates(subset=['short_name']).reset_index(drop=True)
df_stations_capacities = df_stations_capacities.rename(columns={'short_name': 'station_id'})

# self.df_station_costs
df_station_costs = df_stations[['station_id']].copy()
df_station_costs['fixed_cost_station'] = 0

# total
df_stations_total = pd.merge(df_stations, df_stations_capacities, on='station_id', how='left')
df_stations_total = pd.merge(df_stations_total, df_station_costs, on='station_id', how='left')
df_stations_total['capacity'] = df_stations_total['capacity'].fillna(df_stations_total['capacity'].mean()).astype(int)

/var/folders/bc/6rm76y3s2f7180l31j70ss7h0000gn/T/ipykernel_55563/3579843473.py:8: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  trips_df =  pd.read_csv("/Users/vladislav/Documents/vlzm/GFDRR/data/raw/202601-citibike-tripdata_1.csv")


In [11]:
import pandas as pd
import requests

GBFS_BASE = "https://gbfs.citibikenyc.com/gbfs/en"
TRIPS_PATH = "E:\\Documents\\vlzm\\GFDRR\\data\\raw\\202602-citibike-tripdata_1.csv"
TRIPS_PATH = "/Users/vladislav/Documents/vlzm/GFDRR/data/raw/202601-citibike-tripdata_1.csv"

def load_trips_df_raw(trips_path: str = TRIPS_PATH) -> pd.DataFrame:
    trips_df = pd.read_csv(trips_path)

    return trips_df

def load_gbfs_raw(gbfs_base: str = GBFS_BASE) -> pd.DataFrame:
    station_info = requests.get(f"{gbfs_base}/station_information.json").json()
    station_status = requests.get(f"{gbfs_base}/station_status.json").json()
    info_df = pd.DataFrame(station_info["data"]["stations"])
    status_df = pd.DataFrame(station_status["data"]["stations"])
    stations = info_df.merge(status_df, on="station_id")
    
    return stations

def get_stations(trips: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for p in ("start", "end"):
        cols = {f"{p}_station_id": "station_id", f"{p}_lat": "lat", f"{p}_lng": "lng"}
        parts.append(trips[list(cols)].rename(columns=cols))

    return (
        pd.concat(parts)
        .dropna(subset=["station_id"])
        .drop_duplicates("station_id")
        .reset_index(drop=True)
    )

def get_stations_capacities(
    gbfs: pd.DataFrame, df_stations: pd.DataFrame
) -> pd.DataFrame:
    return (
        gbfs.query("is_installed == 1")[["short_name", "capacity"]]
        .drop_duplicates("short_name")
        .rename(columns={"short_name": "station_id"})
        .query("station_id in @df_stations.station_id")
        .reset_index(drop=True)
    )

def get_stations_costs(df_stations: pd.DataFrame) -> pd.DataFrame:
    return df_stations[["station_id"]].assign(fixed_cost_station=0)

def get_depots(rng: np.random.Generator, n: int) -> pd.DataFrame:
    return pd.DataFrame({
        "node_id": [f"depot_{i + 1}" for i in range(n)],
        "lat":     rng.uniform(40.68, 40.86, size=n),
        "lon":     rng.uniform(-74.03, -73.90, size=n),
    })

def get_depots_capacities(
    rng: np.random.Generator, depot_capacity: int, df_depots: pd.DataFrame
) -> pd.DataFrame:
    return df_depots[["node_id"]].assign(
        capacity=rng.integers(
            int(depot_capacity * 0.75), int(depot_capacity * 1.25), size=len(df_depots)
        )
    )

def get_depots_costs(rng: np.random.Generator, df_depots: pd.DataFrame) -> pd.DataFrame:
    return df_depots[["node_id"]].assign(
        fixed_cost_depot=np.round(rng.uniform(80.0, 200.0, size=len(df_depots)), 2)
    )

def get_timestamps(trips_df_raw):
    start_date = pd.to_datetime(trips_df_raw['started_at']).min().floor('h')
    end_date = pd.to_datetime(trips_df_raw['started_at']).max().ceil('h')
    timestamps = pd.date_range(start=start_date, end=end_date, freq="h")
    
    return timestamps

_rng = np.random.default_rng(seed=42)
n = 10
depot_capacity = 9000
ebike_fraction = 0.3

trips_df_raw = load_trips_df_raw(trips_path=TRIPS_PATH)
gbfs_raw = load_gbfs_raw(GBFS_BASE)

df_stations = get_stations(trips_df_raw)
df_stations_capacities = get_stations_capacities(gbfs_raw, df_stations)
df_stations_costs = get_stations_costs(df_stations)

df_depots = get_depots(_rng, n)
df_depot_capacities = get_depots_capacities(_rng, depot_capacity, df_depots)
df_depot_costs = get_depots_costs(_rng, df_depots)

timestamps = get_timestamps(trips_df_raw)

### ATTENTION!!! 
# I think we also can add number of depots as parameter for simulator to check how it will influence the solution.
# Additionally, we can manualy set locations of depots to check how it will influence the solution. 
# For example, we can set one depot in the center of the city and second depot in the periphery. 
# It will be interesting to see how it will influence the solution.

/var/folders/bc/6rm76y3s2f7180l31j70ss7h0000gn/T/ipykernel_55563/98751316.py:9: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  trips_df = pd.read_csv(trips_path)


,station_id,num_bikes_available
0,4743.04,0
1,5216.06,0
2,6560.01,0
3,5788.15,0
4,4775.01,0
...,...,...
2198,HB106,3
2199,HB201,27
2200,HB603,8
2201,HB608,10


In [ ]:
# For inventory use stations
# self.inventory_initial,

inventory_initial = gbfs_raw[['short_name','num_bikes_available']].copy()
inventory_initial = inventory_initial.rename(columns={'short_name': 'station_id', 'num_bikes_available': 'num_bikes_available'})
inventory_initial = inventory_initial.query("station_id in @df_stations.station_id").reset_index(drop=True)
inventory_initial

In [ ]:
# For trips data use trips data
# self.df_trips

In [ ]:
# For telemetry use stations
# self.inventory_initial,
# self.df_telemetry_ts, Это я могу накопить. Но так же я могу использовать inventory_initial и trips для восстановления df_telemetry_ts

In [ ]:
# For resources use mock functions
# self.df_resources
# self.df_resource_capacities
# self.df_truck_rates